# Class 8 - Contextualized Embeddings
## The Case of Word Sense Disambiguity
Word sense disambiguation (WSD) is the task of determining which meaning of a word is intended in context. The classic example we are gonna work with today is the lemma 'bank'. I have generated a corpus with Claude of 100 sentences. 50 sentences describe a river bank. 50 sentences describe a financial bank. Traditional methods relied on hand-crafted features or lexical resources, but modern transformer-based models instead represent meaning through contextual embeddings.

We are gonna use sentence transformers, as that library is very easy to utilize. The plan is as follows:
1. Get sentence embeddings for all sentences.
2. Find the average word sense of bank in both cases.
3. Find tokens that are close to the word sense.
4. Make a PCA-reduced plot of your results.

**Extra reading:**

Teglia, S., Tedeschi, S., & Navigli, R. (2025, July). How Much Do Encoder Models Know About Word Senses?. In Proceedings of the 63rd Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers) (pp. 2266-2277).
https://aclanthology.org/2025.acl-long.113.pdf

*TLDR:* Historical introduction to word sense disambiguity in CompLing, finds that the intermediate layers of LLMs encode word senses more accurately.

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn sentence-transformers tqdm
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [ ]:
# Load the data:
df = pd.read_csv("../data/bank_sentences.csv")
# Show 3 first and 3 last rows in one cell:
df.iloc[[0, 1, 2, 97, 98, 99]]

,Number,Type,Sentence
0,1,Financial Bank,She deposited her paycheck at the bank on Frid...
1,2,Financial Bank,The bank approved her mortgage application aft...
2,3,Financial Bank,He stood in line at the bank for nearly thirty...
97,98,River Bank,He wrote her name in the soft mud of the bank ...
98,99,River Bank,The ancient willows had grown so large their r...
99,100,River Bank,She promised herself she would return to that ...


# Get embeddings of the sentences:
While we are using mean-pooled sentence embeddings. The only word that consistently appear in all sentences is bank. On average the word bank will by far have the largest influence on the sentence embeddings, if we use the average sentence embedding across multiple sentences.

In [33]:
# This is the model we will use:
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu") # Use CPU to avoid GPU issues

# These are our sentences:
sentences = df["Sentence"].tolist()

# Figure out how to get the embeddings by reading the documentation of the model.
# https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2


# Add the embeddings to the dataframe (it prefers a list format):



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Figure out what word sense is most similar to non-contexualized bank:

In [ ]:
# Now we can do some analysis! For example, let's compute the mean embedding for each type of bank and see how similar they are:
# Compute the mean embedding of the 50 sentences of each type:

# Find the embedding of the word "bank", by treating it as a sentence and passing it through the model. (This is a bit hacky, but it works for this model.)

# What word sense is the most similar to the word "bank"?:


# Check similarity of other words, to see if the word sense fits:

In [ ]:
# Candidate vocabulary (can expand!)
vocab = [
    "money", "cash", "loan", "finance", "river", "water",
    "shore", "stream", "investment", "credit", "pond",
    "sand", "rain", "account", "deposit", "stream", "flood"
]

vocab_embs = model.encode(vocab)

# Create a function that finds the top k most similar words in the vocab to a target vector, using cosine similarity or distance:
# The function should:
# 1. Compute the cosine similarity / distance between the target vector and all vocab embeddings.
# 2. Sort the vocab words by similarity / distance
# 3. Return the top k most similar words and their similarity scores.

    
# Use the function to display most similar words to river_mean and finance_mean embeddings, and see if they make sense!:



# Reduce dimensions with PCA and plot a scatterplot:
(Nice visuals is always important!)

In [ ]:
# Now let's do PCA to visualize the sentence embeddings and the vocab embeddings in 2D:
# Remember this is how you use PCA in sklearn: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
# from sklearn.decomposition import PCA
# pca = PCA(n_components=2)
# reduced = pca.fit_transform(your_embeddings)





In [ ]:
# Plot the sentence embeddings in 2D, colored by their label (river vs finance):
# You can use the seaborn scatterplot function for this:
# sns.scatterplot(x="pc1", y="pc2", hue="label", data=df)
# https://www.geeksforgeeks.org/python/scatterplot-using-seaborn-in-python/




# Extra task: Can you get the token "vocab_embs" to show up in the same plot as the sentence embeddings? They need to be reduced to 2D using the same PCA transformation as the sentence embeddings.

## Extra: Token Level Analysis
If you want to try the complete the analysis with the token-level representations of bank, i have written a function to get the bank tokens. The code is a bit more complex, so im importing a helper function from helper.py.
You can always check it out if you need it at a later stage.

*OBS:* The token level representations a bit more gimmicky

In [32]:
from helper import get_bank_embedding
embeddings = []
for sent in sentences:
         # Imported function from helper.py that extracts the embedding of the target word in the sentence
         emb = get_bank_embedding(sent, model, TARGET_WORD)
         embeddings.append(emb)
